In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
def create_sparkSession():
    spark = SparkSession.builder.appName("PatientVitalsFileStreaming").getOrCreate()
    return spark    

In [ ]:
def get_schema():
    schema = StructType([
        StructField("patient_id", StringType()),
        StructField("event_time", StringType()),  # Read as STRING
        StructField("bp_systolic", IntegerType()),
        StructField("bp_diastolic", IntegerType()),
        StructField("temperature", DoubleType()),
        StructField("heart_rate", IntegerType()),
        StructField("spo2", IntegerType())
    ])

    return schema
    

In [ ]:
spark = create_sparkSession()
schema = get_schema()

In [ ]:
vitals_df = spark.readStream \
    .format("json") \
    .schema(schema) \
    .option("maxFilesPerTrigger", 1) \
    .load("./data/") \
    .withColumn("event_time",to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss")) \
    .withWatermark("event_time", "1 minute")

## Critical Patient Alert

In [ ]:
critical_alerts = vitals_df.filter(
    (col("temperature") > 102) |
    (col("heart_rate") > 120) |
    (col("spo2") < 90)
)

query1 = critical_alerts.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .start()


## Sliding Window Avg Heart Rate

In [ ]:

avg_hr = vitals_df.groupBy(window(col("event_time"), "1 minute", "30 seconds"),col("patient_id"))\
    .agg(avg("heart_rate").alias("avg_heart_rate"))

query2 = avg_hr.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .start()


## Max BP per patient

In [ ]:
max_bp = vitals_df.groupBy("patient_id") \
    .agg(max("bp_systolic").alias("max_systolic_bp"))

query3 = max_bp.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .start()


## Abnormal count for 2-min window

In [ ]:
abnormal = vitals_df.filter((col("temperature") > 101) | (col("spo2") < 92))

abnormal_count = abnormal.groupBy(
    window(col("event_time"), "2 minutes"),col("patient_id")).count()

query4 = abnormal_count.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .start()


## Running average temperature per patient

In [ ]:
running_avg_temp = vitals_df.groupBy("patient_id") \
    .agg(avg("temperature").alias("avg_temperature"))

query5 = running_avg_temp.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .start()
